In [1]:
import pandas as pd
import string
import unicodedata
from langdetect import detect, LangDetectException
import os
import re

In [2]:
def is_non_english(text):
    """
    https://github.com/knoveleng/open-rs/blob/main/src/open_r1/rewards.py
    Checks if the given text contains languages other than English.
    Ignores LaTeX notation.
    
    Args:
        text (str): The text to analyze
        
    Returns:
        bool: False if the text is in English (with LaTeX allowed),
            True if it contains non-English languages
    """
    # Skip if empty
    if not text or text.strip() == "":
        return False
    
    # First, remove LaTeX notation to avoid false positives
    # This pattern matches typical LaTeX structures like $...$ or \begin{...}...\end{...}
    # latex_pattern = r'\$[^$]*\$|\\\(.*?\\\)|\\\[.*?\\\]|\\begin\{.*?\}.*?\\end\{.*?\}'
    # text_without_latex = re.sub(latex_pattern, '', text, flags=re.DOTALL)
    
    # # Also remove common LaTeX commands
    # latex_commands = r'\\[a-zA-Z]+((\{[^{}]*\})?|(\[[^\[\]]*\])?)+'
    # text_without_latex = re.sub(latex_commands, '', text_without_latex)
    
    # Check if we have non-ASCII characters that are not typical in English text
    # First, normalize unicode characters
    normalized_text = unicodedata.normalize('NFKD', text)
    
    # Common non-English character sets (excluding common punctuation and symbols)
    non_english_patterns = [
        # Cyrillic characters
        r'[\u0400-\u04FF]',
        # Chinese/Japanese/Korean characters
        r'[\u3040-\u30FF\u3400-\u4DBF\u4E00-\u9FFF\uF900-\uFAFF]',
        # Arabic characters
        r'[\u0600-\u06FF]',
        # Hebrew characters
        r'[\u0590-\u05FF]',
        # Thai characters
        r'[\u0E00-\u0E7F]',
        # Greek characters
        # r'[\u0370-\u03FF]',
    ]
    
    for pattern in non_english_patterns:
        if re.search(pattern, normalized_text):
            return True
    
    # If no obvious non-English characters found, try language detection
    # Clean text further - remove URLs, numbers, punctuation
    cleaned_text = re.sub(r'http\S+|www\S+|\d+|[^\w\s]', ' ', text)
    cleaned_text = ' '.join(cleaned_text.split())
    # print(cleaned_text)
    
    # Only perform language detection if we have enough text
    if len(cleaned_text.split()) >= 5:
        try:
            detected_lang = detect(cleaned_text)
            # print(detected_lang)
            return detected_lang != 'en'
        except LangDetectException("LangDetectException"):
            # If detection fails, rely on character-based detection above
            pass
    
    # Default to assuming it's English
    return False

In [3]:
directory_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/adrd_simplified_evaluation/llm_answer_extractor/extracted_results_sub/Neuropath"

In [4]:
csv_files = {}
files = os.listdir(directory_path)
for file in files:
    if os.path.isfile(os.path.join(directory_path, file)):
        csv_files[file] = pd.read_csv(os.path.join(directory_path,file))

In [8]:
from tqdm import tqdm

# Enable tqdm for pandas apply
tqdm.pandas()

def check_non_english_text(row):
    text = row['generated_text']
    row["non_english"] = is_non_english(text)
    return row


In [9]:
# Apply the function with progress bar
csv_files = {
    key: df.progress_apply(check_non_english_text, axis=1)
    for key, df in csv_files.items()
}

100%|██████████| 10688/10688 [00:53<00:00, 201.23it/s]


In [10]:
non_english_proportions = {}
for key, value in csv_files.items():
    non_english_dict = dict(value['non_english'].value_counts())
    non_english_proportion = non_english_dict[True] / len(value)
    # print(key, non_english_proportion)
    non_english_proportions[key] = round(non_english_proportion, 3)

In [11]:
dict(sorted(non_english_proportions.items(), key=lambda item: item[1]))

{'qwen25_3B_drgrpo_gp16_all_train_sub_old_prompt_Neuropath.csv': 0.006,
 'qwen25_3B_old_prompt_Neuropath.csv': 0.01,
 'qwen25_3B_drgrpo_gp16_all_train_sub_Neuropath.csv': 0.012,
 'qwen25_3B_Neuropath.csv': 0.015}

In [13]:
print(csv_files['qwen25_3B_Neuropath.csv'][csv_files['qwen25_3B_Neuropath.csv']['non_english']].iloc[0]['generated_text'])

Let's analyze the provided information step by step:

1. **Demographic and Referral Details**:
   - Age: 95
   - Ethnicity: Multiracial (Black or African American with American Indian or Alaska Native, and White)
   - Education: Completed 18 years
   - Language: English
   - Marital status: Currently married, but currently relies entirely on others (widowed with a co-participant who is a child)

2. **Primary Cognitive Assessment**—The Neuropsychiatric Inventory Questionnaire (NPI-Q) reports mild delusions but no hallucinations, agitation, depression, anxiety, elation, or noticeable changes in apathy or disinhibition.

3. **Functional Assessment**:
   - Decline in functionality across multiple domains (writing checks, taxes, shopping, finances, independent living)

4. **Current Medications**:
   - LATANOPROST OPHTHALMIC (prescribed for glaucoma, which is a common non-neurodegenerative condition)

5. **Clinical Neurological Exclusion**:
   - Does not have any of several other severe neur